# data.table

In [1]:
# Imports

library(NHANES)
library(data.table)
library(dslabs)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   3.5.1     ✔ tibble    3.3.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::between()     masks data.table::between()
✖ dplyr::filter()      masks stats::filter()
✖ dplyr::first()       masks data.table::first()
✖ lubridate::hour()    masks data.table::hour()
✖ lubridate::isoweek() masks data.table::isoweek()
✖ dplyr::lag()         masks stats::lag()
✖ dplyr::last()        masks data.table::last()
✖ lubridate::mday()    masks data.table::mday()
✖ lubridate::minute()  masks data.table::minute()
✖ lubridate::month()   masks data.table::month()
✖ lubridate::quarter() masks data.table::quarter()
✖ lubridate::second()  masks data.table::second()
✖ purrr::transpose()   masks data.table::transpose()
✖ lubridate::wday() 

The book will focus on Tidyverse packages for ease of use, though in this chapter the library data.table will be described a bit. The way the book frames things, there is a tradeoff between the readability and ease of use of the Tidyverse and the efficiency and speed of data.table, though this appears to be somewhat of a false dichotomy and there are efforts like [tidytable](https://markfairbanks.github.io/tidytable/) that attempt to bridge the gap. For now however we are only looking at introductory material.

## Refining data tables

In order to operate on data tables, we will need one first:

In [2]:
murders_dt <- as.data.table(murders)
head(murders_dt)

state,abb,region,population,total
<chr>,<chr>,<fct>,<dbl>,<dbl>
Alabama,AL,South,4779736,135
Alaska,AK,West,710231,19
Arizona,AZ,West,6392017,232
Arkansas,AR,South,2915918,93
California,CA,West,37253956,1257
Colorado,CO,West,5029196,65


### Column-wise subsetting

Selecting with data tables is done similar to subsetting matrices. Whereas with dplyr one would write, for example:

In [3]:
murders |> select(state, region) |> head()

,state,region
,<chr>,<fct>
1,Alabama,South
2,Alaska,West
3,Arizona,West
4,Arkansas,South
5,California,West
6,Colorado,West


With data tables, one possible way of subsetting goes like so:

In [4]:
head(murders_dt[, c('state', 'region')])

state,region
<chr>,<fct>
Alabama,South
Alaska,West
Arizona,West
Arkansas,South
California,West
Colorado,West


The `.()` function can also be used to make it clear that unquoted words are not intended to be part of the overall environment:

In [5]:
head(murders_dt[, .(state, region)])

state,region
<chr>,<fct>
Alabama,South
Alaska,West
Arizona,West
Arkansas,South
California,West
Colorado,West


### Adding or transforming variables

With dplyr, `mutate()` works like this:

In [6]:
murders <- murders |> mutate(rate = total / population * 10^5)
head(murders)

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135,2.824424
2,Alaska,AK,West,710231,19,2.675186
3,Arizona,AZ,West,6392017,232,3.629527
4,Arkansas,AR,South,2915918,93,3.189390
5,California,CA,West,37253956,1257,3.374138
6,Colorado,CO,West,5029196,65,1.292453


To conserve memory, data.table doesn't copy on mutation but rather changes things *in situ*:

In [7]:
murders_dt[, rate := total / population * 10^5]
head(murders_dt)

state,abb,region,population,total,rate
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
Alabama,AL,South,4779736,135,2.824424
Alaska,AK,West,710231,19,2.675186
Arizona,AZ,West,6392017,232,3.629527
Arkansas,AR,South,2915918,93,3.189390
California,CA,West,37253956,1257,3.374138
Colorado,CO,West,5029196,65,1.292453


It is possible to define multiple new columns at once:

In [8]:
murders_dt[, ':='(rate = total / population * 10^5, rank = rank(population))]
head(murders_dt)

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Alabama,AL,South,4779736,135,2.824424,29
Alaska,AK,West,710231,19,2.675186,5
Arizona,AZ,West,6392017,232,3.629527,36
Arkansas,AR,South,2915918,93,3.189390,20
California,CA,West,37253956,1257,3.374138,51
Colorado,CO,West,5029196,65,1.292453,30


### Reference versus copy

I'll just crib from the book here:

"The data.table package is designed to avoid wasting memory. So if you make a copy of a table, like this:"

In [9]:
x <- data.table(a = 1)
y <- x

"`y` is actually referencing `x`, it is not an new object: `y` just another name for `x`. Until you change `y`, a new object will not be made. However, the `:=` function changes by reference so if you change `x`, a new object is not made and `y` continues to be just another name for `x`:"

In [10]:
x[, a := 2]
y

a
<dbl>
2


"You can also change `x` like this:"

In [11]:
y[, a := 1]
x

a
<dbl>
1


The data.table function `copy()` deviates from this default behavior:

In [12]:
x <- data.table(a = 1)
y <- copy(x)
x[, a := 2]
y

a
<dbl>
1


It's worth noting that `as.data.table()` creates a copy of the original object (which can be a data frame, tibble as well as a few others). When such behavior is not desired due to memory limitations, the object can be modified *in situ* like so:

In [13]:
x <- data.frame(a = 1)
setDT(x)
x

a
<dbl>
1


Because no copy is being made the following code does not create a new object:

In [14]:
x <- data.frame(a = 1)
y <- setDT(x)
x[, a := 2]
y

a
<dbl>
2


### Row-wise subsetting

With dplyr we filtered rows like so:

In [15]:
murders |> filter(rate <= 0.7)

state,abb,region,population,total,rate
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
Hawaii,HI,West,1360301,7,0.5145920
Iowa,IA,North Central,3046355,21,0.6893484
New Hampshire,NH,Northeast,1316470,5,0.3798036
North Dakota,ND,North Central,672591,4,0.5947151
Vermont,VT,Northeast,625741,2,0.3196211


data.table also uses non-standard evaluation which means that unquoted strings are not treated like references to objects in the environment:

In [16]:
murders_dt[rate <= 0.7]

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Hawaii,HI,West,1360301,7,0.5145920,12
Iowa,IA,North Central,3046355,21,0.6893484,22
New Hampshire,NH,Northeast,1316470,5,0.3798036,10
North Dakota,ND,North Central,672591,4,0.5947151,4
Vermont,VT,Northeast,625741,2,0.3196211,3


The equivalents of `filter()` and `select()` can be done in the same operation:

In [17]:
murders_dt[rate <= 0.7, .(state, rate)]

state,rate
<chr>,<dbl>
Hawaii,0.5145920
Iowa,0.6893484
New Hampshire,0.3798036
North Dakota,0.5947151
Vermont,0.3196211


## Summarizing data

As an example, we will use the `heights` dataset:

In [18]:
heights_dt <- as.data.table(heights)
head(heights_dt)

sex,height
<fct>,<dbl>
Male,75
Male,70
Male,68
Male,74
Male,61
Female,65


Here is an example of summarizing the original dataset with dplyr:

In [19]:
heights |> summarize(avg = mean(height), sd = sd(height))

avg,sd
<dbl>,<dbl>
68.32301,4.078617


Here is the following using data.table, using `.()` to apply functions to columns:

In [20]:
heights_dt[, .(avg = mean(height), sd = sd(height))]

avg,sd
<dbl>,<dbl>
68.32301,4.078617


Note that filtering and summarizing can be done compactly in one step. Instead of:

In [21]:
heights |> filter(sex == 'Female') |> summarize(avg = mean(height), sd = sd(height))

avg,sd
<dbl>,<dbl>
64.93942,3.760656


data.table uses the following syntax:

In [22]:
heights_dt[sex == 'Female', .(avg = mean(height), sd = sd(height))]

avg,sd
<dbl>,<dbl>
64.93942,3.760656


### Multiple summaries

Remember the following function back from chapter 4?

In [23]:
median_min_max <- function(x) {
    qs <- quantile(x, c(0.5, 0, 1))
    data.frame(median = qs[1], minimum = qs[2], maximum = qs[3])
}

Packaged data frame summaries are also possibly with data.table:

In [24]:
heights_dt[, .(median_min_max(height))]

median,minimum,maximum
<dbl>,<dbl>,<dbl>
68.5,50,82.67717


### Group then summarize

All you need is the `by` keyword for grouping by categorical variables:

In [25]:
heights_dt[, .(avg = mean(height), sd = sd(height)), by = sex]

sex,avg,sd
<fct>,<dbl>,<dbl>
Male,69.31475,3.611024
Female,64.93942,3.760656


## Sorting

Sorting syntax is a bit similar to filtering:

In [26]:
murders_dt[order(population)]

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Wyoming,WY,West,563626,5,0.8871131,1
District of Columbia,DC,South,601723,99,16.4527532,2
Vermont,VT,Northeast,625741,2,0.3196211,3
North Dakota,ND,North Central,672591,4,0.5947151,4
Alaska,AK,West,710231,19,2.6751860,5
South Dakota,SD,North Central,814180,8,0.9825837,6
Delaware,DE,South,897934,38,4.2319369,7
Montana,MT,West,989415,12,1.2128379,8
Rhode Island,RI,Northeast,1052567,16,1.5200933,9


Descending sorts can be done in one of two ways, first with the negation of the column:

In [27]:
murders_dt[order(-population)]

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
California,CA,West,37253956,1257,3.3741383,51
Texas,TX,South,25145561,805,3.2013603,50
Florida,FL,South,19687653,669,3.3980688,49
New York,NY,Northeast,19378102,517,2.6679599,48
Illinois,IL,North Central,12830632,364,2.8369608,47
Pennsylvania,PA,Northeast,12702379,457,3.5977513,46
Ohio,OH,North Central,11536504,310,2.6871225,45
Georgia,GA,South,9920000,376,3.7903226,44
Michigan,MI,North Central,9883640,413,4.1786225,43


A more verbose alternative is available too:

In [28]:
murders_dt[order(population, decreasing = TRUE)]

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
California,CA,West,37253956,1257,3.3741383,51
Texas,TX,South,25145561,805,3.2013603,50
Florida,FL,South,19687653,669,3.3980688,49
New York,NY,Northeast,19378102,517,2.6679599,48
Illinois,IL,North Central,12830632,364,2.8369608,47
Pennsylvania,PA,Northeast,12702379,457,3.5977513,46
Ohio,OH,North Central,11536504,310,2.6871225,45
Georgia,GA,South,9920000,376,3.7903226,44
Michigan,MI,North Central,9883640,413,4.1786225,43


Nested ordering is also possible as before (note that ordering of the `region` isn't automatically done alphabetically but rather in accord with the idiosyncratic ordering of that factor):

In [29]:
murders_dt[order(region, rate)]

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Vermont,VT,Northeast,625741,2,0.3196211,3
New Hampshire,NH,Northeast,1316470,5,0.3798036,10
Maine,ME,Northeast,1328361,11,0.8280881,11
Rhode Island,RI,Northeast,1052567,16,1.5200933,9
Massachusetts,MA,Northeast,6547629,118,1.8021791,38
New York,NY,Northeast,19378102,517,2.6679599,48
Connecticut,CT,Northeast,3574097,97,2.7139722,23
New Jersey,NJ,Northeast,8791894,246,2.7980319,41
Pennsylvania,PA,Northeast,12702379,457,3.5977513,46


## Exercises

**Exercise 1**: Turn `murders` into `murders_dt`, then add a column called `rate` with homicides per 100,000:

In [30]:
murders_dt <- as.data.table(murders)
murders_dt[, rate := total / population * 10^5]
head(murders_dt)

state,abb,region,population,total,rate
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
Alabama,AL,South,4779736,135,2.824424
Alaska,AK,West,710231,19,2.675186
Arizona,AZ,West,6392017,232,3.629527
Arkansas,AR,South,2915918,93,3.189390
California,CA,West,37253956,1257,3.374138
Colorado,CO,West,5029196,65,1.292453


**Exercise 2**: Add a column called `rank` with the highest to lowest homicide rates ranked:

In [31]:
murders_dt[, rank := rank(-rate)]
murders_dt

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Alabama,AL,South,4779736,135,2.8244238,23
Alaska,AK,West,710231,19,2.6751860,27
Arizona,AZ,West,6392017,232,3.6295273,10
Arkansas,AR,South,2915918,93,3.1893901,17
California,CA,West,37253956,1257,3.3741383,14
Colorado,CO,West,5029196,65,1.2924531,38
Connecticut,CT,Northeast,3574097,97,2.7139722,25
Delaware,DE,South,897934,38,4.2319369,6
District of Columbia,DC,South,601723,99,16.4527532,1


**Exercise 3**: If we want to only show the states and population sizes, we can use:

In [32]:
head(murders_dt[, .(state, population)])

state,population
<chr>,<dbl>
Alabama,4779736
Alaska,710231
Arizona,6392017
Arkansas,2915918
California,37253956
Colorado,5029196


Show the state names and abbreviations in murders.

In [33]:
head(murders_dt[, .(state, abb)])

state,abb
<chr>,<chr>
Alabama,AL
Alaska,AK
Arizona,AZ
Arkansas,AR
California,CA
Colorado,CO


**Exercise 4**: You can show just the New York row like this:

In [34]:
murders_dt[state == 'New York']

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
New York,NY,Northeast,19378102,517,2.66796,29


You can use other logical vectors to filter rows.

Show the top 5 states with the highest murder rates. From here on, do not change the `murders` dataset, just show the result. Remember that you can filter based on the `rank` column.

In [35]:
murders_dt[rank <= 5]

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
District of Columbia,DC,South,601723,99,16.452753,1
Louisiana,LA,South,4533372,351,7.742581,2
Maryland,MD,South,5773552,293,5.074866,4
Missouri,MO,North Central,5988927,321,5.359892,3
South Carolina,SC,South,4625364,207,4.475323,5


**Exercise 5**: We can remove rows using the `!=` operator. For example, to remove Florida, we would do this:

In [36]:
no_florida <- murders_dt[state != 'Florida']

Create a new data frame called `no_south` that removes states from the South region. How many states are in this category? You can use the function `nrow()` for this.

In [37]:
no_south <- murders_dt[region != 'South']
no_south

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Alaska,AK,West,710231,19,2.6751860,27
Arizona,AZ,West,6392017,232,3.6295273,10
California,CA,West,37253956,1257,3.3741383,14
Colorado,CO,West,5029196,65,1.2924531,38
Connecticut,CT,Northeast,3574097,97,2.7139722,25
Hawaii,HI,West,1360301,7,0.5145920,49
Idaho,ID,West,1567582,12,0.7655102,46
Illinois,IL,North Central,12830632,364,2.8369608,22
Indiana,IN,North Central,6483802,142,2.1900730,31


In [38]:
nrow(no_south)

[1] 34

**Exercise 6**: We can also use `%in%` to filter. You can therefore see the data from New York and Texas as follows:

In [39]:
murders_dt[state %in% c('New York', 'Texas')]

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
New York,NY,Northeast,19378102,517,2.66796,29
Texas,TX,South,25145561,805,3.20136,16


Create a new data frame called `murders_nw` with only the states from the Northeast and the West. How many states are in this category?

In [40]:
murders_nw <- murders_dt[region %in% c('Northeast', 'West')]
murders_nw

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Alaska,AK,West,710231,19,2.6751860,27
Arizona,AZ,West,6392017,232,3.6295273,10
California,CA,West,37253956,1257,3.3741383,14
Colorado,CO,West,5029196,65,1.2924531,38
Connecticut,CT,Northeast,3574097,97,2.7139722,25
Hawaii,HI,West,1360301,7,0.5145920,49
Idaho,ID,West,1567582,12,0.7655102,46
Maine,ME,Northeast,1328361,11,0.8280881,44
Massachusetts,MA,Northeast,6547629,118,1.8021791,32


In [41]:
nrow(murders_nw)

[1] 22

**Exercise 7**: Suppose you want to live in the Northeast or West and want the murder rate to be less than 1. We want to see the data for the states satisfying these options. Note that you can use logical operators with filter. Here is an example in which we filter to keep only small states in the Northeast region:

In [42]:
murders_dt[population < 5000000 & region == 'Northeast']

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Connecticut,CT,Northeast,3574097,97,2.7139722,25
Maine,ME,Northeast,1328361,11,0.8280881,44
New Hampshire,NH,Northeast,1316470,5,0.3798036,50
Rhode Island,RI,Northeast,1052567,16,1.5200933,35
Vermont,VT,Northeast,625741,2,0.3196211,51


Make sure `murders` has been defined with `rate` and `rank` and still has all states. Create a table called `my_states` that contains rows for states satisfying both the conditions: they are in the Northeast or West and the murder rate is less than 1. Show only the state name, the rate, and the rank.

In [43]:
my_states <- murders_dt[region %in% c('Northeast', 'West') & rate <= 1, .(state, rate, rank)]
my_states

state,rate,rank
<chr>,<dbl>,<dbl>
Hawaii,0.5145920,49
Idaho,0.7655102,46
Maine,0.8280881,44
New Hampshire,0.3798036,50
Oregon,0.9396843,42
Utah,0.7959810,45
Vermont,0.3196211,51
Wyoming,0.8871131,43


Exercises 8 through 12 will be using the NHANES library already imported above:

**Exercise 8**: We will provide some basic facts about blood pressure. First let's select a group to set the standard. We will use 20-to-29-year-old females. `AgeDecade` is a categorical variable with these ages. Note that the category is coded like `' 20-29'`, with a space in front! Use the data.table package to compute the average and standard deviation of systolic blood pressure as saved in the `BPSysAve` variable. Save it to a variable called `ref`.

In [44]:
ref <- as.data.table(NHANES)[
    AgeDecade == ' 20-29' & Gender == 'female',
    .(avg = mean(BPSysAve, na.rm = TRUE), sd = sd(BPSysAve, na.rm = TRUE))
]

ref

avg,sd
<dbl>,<dbl>
108.4224,10.14668


**Exercice 9**: Report the min and max values for the same group.

In [45]:
as.data.table(NHANES)[
    AgeDecade == ' 20-29' & Gender == 'female',
    .(min = min(BPSysAve, na.rm = TRUE), max = max(BPSysAve, na.rm = TRUE))
]

min,max
<int>,<int>
84,179


**Exercise 10**: Compute the average and standard deviation for females, but for each age group separately rather than a selected decade as in exercise 8. Note that the age groups are defined by `AgeDecade`.

In [46]:
as.data.table(NHANES)[
    Gender == 'female',
    .(avg = mean(BPSysAve, na.rm = TRUE), sd = sd(BPSysAve, na.rm = TRUE)),
    by = AgeDecade
]

AgeDecade,avg,sd
<fct>,<dbl>,<dbl>
40-49,115.49385,14.530054
10-19,104.27466,9.461431
50-59,121.84245,16.179333
0-9,99.95041,9.071798
60-69,127.17787,17.125713
20-29,108.42243,10.146681
30-39,111.25512,12.314790
70+,133.51652,19.841781
NA,141.54839,22.908521


**Exercise 11**: Repeat exercise 10 for males.

In [47]:
as.data.table(NHANES)[
    Gender == 'male',
    .(avg = mean(BPSysAve, na.rm = TRUE), sd = sd(BPSysAve, na.rm = TRUE)),
    by = AgeDecade
]

AgeDecade,avg,sd
<fct>,<dbl>,<dbl>
30-39,119.40063,12.306656
0-9,97.41912,8.317367
60-69,126.88578,17.478117
50-59,125.75000,17.760536
10-19,109.59789,11.227769
40-49,120.78390,13.968338
20-29,117.85084,11.274795
NA,136.40000,23.534731
70+,130.20172,18.657475


**Exercise 12**: For males between the ages of 40-49, compare systolic blood pressure across race as reported in the Race1 variable. Order the resulting table from lowest to highest average systolic blood pressure.

In [48]:
racial_averages <- as.data.table(NHANES)[
    Gender == 'male' & AgeDecade == ' 40-49',
    .(avg = mean(BPSysAve, na.rm = TRUE)),
    by = Race1
]

racial_averages[order(avg)]

Race1,avg
<fct>,<dbl>
White,119.9188
Other,120.4000
Hispanic,121.6098
Mexican,121.8500
Black,125.8387
